In [1]:
import numpy as np, random, tensorflow as tf
from collections import deque
from enum import Enum
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from IPython.display import clear_output, display
import time

In [2]:
class Direction(Enum):
    RIGHT = 0; DOWN = 1; LEFT = 2; UP = 3

class SnakeEnv:
    def __init__(self, size=20):
        self.size = size
        self.reset()

    def reset(self):
        self.direction = Direction.RIGHT
        mid = self.size // 2
        self.head = [mid, mid]
        self.tail = [[mid - 1, mid]]
        self._place_fruit()
        self.score = 0
        self.frame = 0
        return self._get_state()

    def render(self, delay=0.1):
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.set_xlim(0, self.size)
        ax.set_ylim(0, self.size)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_aspect('equal')

        fx, fy = self.fruit
        fruit_patch = patches.Rectangle((fx, self.size - fy - 1), 1, 1, color='red')
        ax.add_patch(fruit_patch)

        hx, hy = self.head
        head_patch = patches.Rectangle((hx, self.size - hy - 1), 1, 1, color='green')
        ax.add_patch(head_patch)

        for x, y in self.tail:
            tail_patch = patches.Rectangle((x, self.size - y - 1), 1, 1, color='lime')
            ax.add_patch(tail_patch)

        clear_output(wait=True)
        display(plt.gcf())
        plt.close()
        time.sleep(delay)

    def step(self, action):
        self._move(action)
        self.frame += 1

        reward, done = 0.1, False  # recompensa base por sobreviver

        # Colisão com parede ou cauda
        if (self.head in self.tail) or not (0 <= self.head[0] < self.size and 0 <= self.head[1] < self.size):
            reward, done = -10, True

        # Comeu a fruta
        elif self.head == self.fruit:
            reward = 30
            self.score += 1
            self.tail.append(self.tail[-1])
            self._place_fruit()

        else:
            # Recompensa por aproximação da fruta
            prev_dist = abs(self.tail[0][0] - self.fruit[0]) + abs(self.tail[0][1] - self.fruit[1])
            new_dist  = abs(self.head[0] - self.fruit[0]) + abs(self.head[1] - self.fruit[1])
            if new_dist < prev_dist:
                reward += 4.0
            else:
                reward -= 1.5

            # Penalidade por demora
            reward -= 0.01

        if not done:
            self.tail.insert(0, self.head.copy())
            self.tail.pop()

        return self._get_state(), reward, done, {"score": self.score}

    def _place_fruit(self):
        positions = set(tuple(p) for p in self.tail + [self.head])
        while True:
            self.fruit = [random.randrange(self.size), random.randrange(self.size)]
            if tuple(self.fruit) not in positions:
                break

    def _move(self, rel_action):
        dir_idx = (self.direction.value + (0, 1, -1)[rel_action]) % 4
        self.direction = Direction(dir_idx)
        x, y = self.head
        if self.direction == Direction.RIGHT:  x += 1
        elif self.direction == Direction.LEFT: x -= 1
        elif self.direction == Direction.DOWN: y += 1
        else:                                  y -= 1
        self.head = [x, y]

    def _get_state(self):
        x, y = self.head
        dir_r = self.direction == Direction.RIGHT
        dir_l = self.direction == Direction.LEFT
        dir_u = self.direction == Direction.UP
        dir_d = self.direction == Direction.DOWN

        def danger(dx, dy):
            nx, ny = x + dx, y + dy
            return (
                nx < 0 or nx >= self.size or ny < 0 or ny >= self.size or
                [nx, ny] in self.tail
            )

        danger_straight = danger( 1 if dir_r else -1 if dir_l else 0,
                                  1 if dir_d else -1 if dir_u else 0)
        danger_right   = danger( 1 if dir_d else -1 if dir_u else 0,
                                 1 if dir_l else -1 if dir_r else 0)
        danger_left    = danger(-1 if dir_d else  1 if dir_u else 0,
                                -1 if dir_l else  1 if dir_r else 0)

        food_left  = self.fruit[0] < x
        food_right = self.fruit[0] > x
        food_up    = self.fruit[1] < y
        food_down  = self.fruit[1] > y

        return np.array([
            danger_straight, danger_right, danger_left,
            dir_l, dir_r, dir_u, dir_d,
            food_left, food_right, food_up, food_down
        ], dtype=np.float32)


In [3]:
class DQNAgent:
    def __init__(self, lr=5e-5, gamma=0.9, mem_size=1500, batch=800):
        self.state_size  = 11
        self.action_size = 3
        self.gamma = gamma
        self.batch_size = batch
        self.memory = deque(maxlen=mem_size)
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = (self.epsilon - self.epsilon_min) / 75
        self.model = self._build(lr)

    def _build(self, lr):
        m = tf.keras.Sequential([
            tf.keras.layers.Input(shape=(self.state_size,)),
            tf.keras.layers.Dense(100, activation='relu'),
            tf.keras.layers.Dense(100, activation='relu'),
            tf.keras.layers.Dense(100, activation='relu'),
            tf.keras.layers.Dense(self.action_size, activation='linear')
        ])
        m.compile(optimizer=tf.keras.optimizers.Adam(lr), loss='mse')
        return m

    def remember(self, s, a, r, s2, done):
        self.memory.append((s, a, r, s2, done))

    def replay(self):
        if len(self.memory) < self.batch_size: return
        minibatch = random.sample(self.memory, self.batch_size)
        states, targets = [], []
        for s, a, r, s2, done in minibatch:
            t = r if done else r + self.gamma * np.max(self.model.predict(s2[None], verbose=0)[0])
            target = self.model.predict(s[None], verbose=0)[0]
            target[a] = t
            states.append(s); targets.append(target)
        self.model.fit(np.array(states), np.array(targets), epochs=1, verbose=0)

    def act(self, state):
        if random.random() < self.epsilon:
            return random.randrange(self.action_size)
        q = self.model.predict(state[None], verbose=0)[0]
        return int(np.argmax(q))

    def update_epsilon(self):
        if self.epsilon > self.epsilon_min:
            self.epsilon -= self.epsilon_decay
            self.epsilon = max(self.epsilon, self.epsilon_min)


In [4]:
def train(episodes=150, render=False, seed=None, agent=None):
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)
        tf.random.set_seed(seed)
        
    env = SnakeEnv()
    
    # Usa agente externo se fornecido, senão cria um novo com batch=800
    if agent is None:
        agent = DQNAgent()

    best_score = 0
    
    for ep in range(1, episodes + 1):
        state = env.reset()
        total_reward, done = False, 0
        
        while not done:
            action = agent.act(state)
            next_state, reward, done, info = env.step(action)

            # Finaliza episódio se ficar muito tempo sem pontuar
            if info["score"] == 0 and env.frame > 1000:
                done = True

            agent.remember(state, action, reward, next_state, done)
            state = next_state
            total_reward += reward

            if render:
                env.render(delay=0.1)

        agent.replay()
        agent.update_epsilon()
        best_score = max(best_score, info["score"])
        print(f"Ep {ep:3d}/{episodes} | score={info['score']:3d} "
              f"| best={best_score:3d} | ε={agent.epsilon:0.3f} | steps={env.frame}")
    
    return agent


In [5]:
def test_agent(agent, episodes=50, render=False, verbose=True):
    env = SnakeEnv()
    scores = []
    
    for ep in range(1, episodes + 1):
        state = env.reset()
        done = False
        total_reward = 0

        while not done:
            action = agent.act(state)
            next_state, reward, done, info = env.step(action)
            state = next_state
            total_reward += reward

            if render:
                env.render(delay=0.1)
        
        scores.append(info["score"])
        if verbose:
            print(f"Test {ep:02d}/{episodes} | score = {info['score']}")

    print("\n--- Test Summary ---")
    print(f"Episodes       : {episodes}")
    print(f"Avg. Score     : {np.mean(scores):.2f}")
    print(f"Max. Score     : {np.max(scores)}")
    print(f"Min. Score     : {np.min(scores)}")
    print(f"Std. Dev       : {np.std(scores):.2f}")
    
    return scores


In [7]:
agent = train(episodes=200, render=False)

Ep   1/200 | score=  0 | best=  0 | ε=0.987 | steps=44
Ep   2/200 | score=  0 | best=  0 | ε=0.974 | steps=31
Ep   3/200 | score=  0 | best=  0 | ε=0.960 | steps=61
Ep   4/200 | score=  0 | best=  0 | ε=0.947 | steps=85
Ep   5/200 | score=  0 | best=  0 | ε=0.934 | steps=92
Ep   6/200 | score=  1 | best=  1 | ε=0.921 | steps=67
Ep   7/200 | score=  0 | best=  1 | ε=0.908 | steps=48
Ep   8/200 | score=  0 | best=  1 | ε=0.894 | steps=74
Ep   9/200 | score=  0 | best=  1 | ε=0.881 | steps=59
Ep  10/200 | score=  0 | best=  1 | ε=0.868 | steps=181
Ep  11/200 | score=  0 | best=  1 | ε=0.855 | steps=247
Ep  12/200 | score=  0 | best=  1 | ε=0.842 | steps=120
Ep  13/200 | score=  0 | best=  1 | ε=0.828 | steps=189
Ep  14/200 | score=  0 | best=  1 | ε=0.815 | steps=168
Ep  15/200 | score=  0 | best=  1 | ε=0.802 | steps=24
Ep  16/200 | score=  0 | best=  1 | ε=0.789 | steps=113
Ep  17/200 | score=  0 | best=  1 | ε=0.776 | steps=54
Ep  18/200 | score=  0 | best=  1 | ε=0.762 | steps=34
Ep  

In [8]:
agent.epsilon = 0.0
test_agent(agent, episodes=100, render=False)

Test 01/100 | score = 7
Test 02/100 | score = 5
Test 03/100 | score = 12
Test 04/100 | score = 3
Test 05/100 | score = 23
Test 06/100 | score = 5
Test 07/100 | score = 5
Test 08/100 | score = 8
Test 09/100 | score = 4
Test 10/100 | score = 17
Test 11/100 | score = 6
Test 12/100 | score = 5
Test 13/100 | score = 10
Test 14/100 | score = 3
Test 15/100 | score = 10
Test 16/100 | score = 4
Test 17/100 | score = 7
Test 18/100 | score = 15
Test 19/100 | score = 3
Test 20/100 | score = 5
Test 21/100 | score = 16
Test 22/100 | score = 11
Test 23/100 | score = 4
Test 24/100 | score = 9
Test 25/100 | score = 10
Test 26/100 | score = 3
Test 27/100 | score = 10
Test 28/100 | score = 3
Test 29/100 | score = 3
Test 30/100 | score = 5
Test 31/100 | score = 3
Test 32/100 | score = 3
Test 33/100 | score = 19
Test 34/100 | score = 5
Test 35/100 | score = 9
Test 36/100 | score = 6
Test 37/100 | score = 8
Test 38/100 | score = 4
Test 39/100 | score = 3
Test 40/100 | score = 5
Test 41/100 | score = 12
Test

[7,
 5,
 12,
 3,
 23,
 5,
 5,
 8,
 4,
 17,
 6,
 5,
 10,
 3,
 10,
 4,
 7,
 15,
 3,
 5,
 16,
 11,
 4,
 9,
 10,
 3,
 10,
 3,
 3,
 5,
 3,
 3,
 19,
 5,
 9,
 6,
 8,
 4,
 3,
 5,
 12,
 4,
 15,
 10,
 4,
 4,
 6,
 10,
 5,
 18,
 3,
 5,
 11,
 5,
 8,
 16,
 3,
 7,
 3,
 9,
 4,
 3,
 23,
 5,
 8,
 6,
 5,
 10,
 4,
 17,
 6,
 12,
 36,
 12,
 6,
 6,
 4,
 25,
 6,
 10,
 10,
 3,
 17,
 5,
 5,
 3,
 6,
 19,
 7,
 3,
 4,
 5,
 4,
 3,
 8,
 28,
 3,
 10,
 7,
 3]